In [1]:
#1. Setup & Sample Data
import pandas as pd

# Platform transactions
platform = pd.DataFrame({
    "txn_id": ["T1", "T2", "T3", "T4", "T5"],
    "date": pd.to_datetime(["2024-01-30", "2024-01-31", "2024-01-31", "2024-01-31", "2024-01-31"]),
    "amount": [100.00, 200.00, 150.50, 300.00, 120.00]
})

# Bank settlements
bank = pd.DataFrame({
    "txn_id": ["T1", "T2", "T3", "T3", "T6"],
    "settlement_date": pd.to_datetime(["2024-01-31", "2024-02-02", "2024-02-01", "2024-02-01", "2024-01-31"]),
    "amount": [100.00, 200.00, 150.49, 150.49, -50.00]
})

In [2]:
#2. Detect Duplicates (Bank)
duplicates = bank[bank.duplicated(subset=["txn_id"], keep=False)]

In [3]:
#3. Merge Data (Reconciliation Base)
merged = pd.merge(
    platform,
    bank,
    on="txn_id",
    how="outer",
    suffixes=("_platform", "_bank"),
    indicator=True
)

In [4]:
#4. Identify Issues

#Missing in Bank
missing_in_bank = merged[merged["_merge"] == "left_only"]
missing_in_bank["issue"] = "Missing Settlement"

#Missing in Platform (Refund / Unknown)
missing_in_platform = merged[merged["_merge"] == "right_only"]
missing_in_platform["issue"] = "Unknown Refund / No Original Txn"

#Amount Mismatch (Rounding)
amount_mismatch = merged[
    (merged["_merge"] == "both") &
    (abs(merged["amount_platform"] - merged["amount_bank"]) > 0.01)
]
amount_mismatch["issue"] = "Rounding Difference"

#Late Settlement
late_settlement = merged[
    (merged["_merge"] == "both") &
    (merged["date"].dt.month != merged["settlement_date"].dt.month)
]
late_settlement["issue"] = "Late Settlement"

#Duplicate Entries
duplicate_issues = duplicates.copy()
duplicate_issues["issue"] = "Duplicate Entry"

C:\Users\DELL\AppData\Local\Temp\ipykernel_12628\3637583678.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_in_bank["issue"] = "Missing Settlement"
C:\Users\DELL\AppData\Local\Temp\ipykernel_12628\3637583678.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_in_platform["issue"] = "Unknown Refund / No Original Txn"
C:\Users\DELL\AppData\Local\Temp\ipykernel_12628\3637583678.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

In [5]:
#5. Combine All Issues

issues = pd.concat([
    missing_in_bank,
    missing_in_platform,
    amount_mismatch,
    late_settlement
], ignore_index=True)

# Add duplicates separately (optional format alignment)
duplicate_issues = duplicate_issues.rename(columns={"amount": "amount_bank"})

In [6]:
#6. Summary Report
summary = issues["issue"].value_counts().reset_index()
summary.columns = ["issue_type", "count"]

print("=== SUMMARY ===")
print(summary)

=== SUMMARY ===
                         issue_type  count
0                   Late Settlement      3
1                Missing Settlement      2
2  Unknown Refund / No Original Txn      1


In [7]:
#7. Detailed Report
print("\n=== DETAILED ISSUES ===")
print(issues[[
    "txn_id",
    "amount_platform",
    "amount_bank",
    "date",
    "settlement_date",
    "issue"
]])


=== DETAILED ISSUES ===
  txn_id  amount_platform  amount_bank       date settlement_date  \
0     T4            300.0          NaN 2024-01-31             NaT   
1     T5            120.0          NaN 2024-01-31             NaT   
2     T6              NaN       -50.00        NaT      2024-01-31   
3     T2            200.0       200.00 2024-01-31      2024-02-02   
4     T3            150.5       150.49 2024-01-31      2024-02-01   
5     T3            150.5       150.49 2024-01-31      2024-02-01   

                              issue  
0                Missing Settlement  
1                Missing Settlement  
2  Unknown Refund / No Original Txn  
3                   Late Settlement  
4                   Late Settlement  
5                   Late Settlement  


In [8]:
#8. (Optional) Total Comparison

platform_total = platform["amount"].sum()
bank_total = bank["amount"].sum()

print("\n=== TOTALS ===")
print(f"Platform Total: {platform_total}")
print(f"Bank Total: {bank_total}")
print(f"Difference: {platform_total - bank_total}")


=== TOTALS ===
Platform Total: 870.5
Bank Total: 550.98
Difference: 319.52


In [9]:
# Tolerance-based matching
TOLERANCE = 0.01

merged["amount_match"] = (
    abs(merged["amount_platform"] - merged["amount_bank"]) <= TOLERANCE
)